In [4]:
import pandas as pd

matchup_code = 'R6CH' # championship matchup
year = 2026

bracket_sims = pd.read_csv('./bracket_simulations.csv')
bracket_sims_w = pd.read_csv('./bracket_simulations_w.csv')

m_bracket_sims = bracket_sims[bracket_sims['Tournament'] == 'M']
w_bracket_sims = bracket_sims_w[bracket_sims_w['Tournament'] == 'W']

seeds_m = pd.read_csv("data/MNCAATourneySeeds.csv")
seeds_w = pd.read_csv("data/WNCAATourneySeeds.csv")

# Manually selecting outcome of first four games
seeds_m = pd.concat([seeds_m, pd.DataFrame([{'Season': 2026, 'Seed': 'X16', 'TeamID': 1250}])], ignore_index=True)
seeds_m = pd.concat([seeds_m, pd.DataFrame([{'Season': 2026, 'Seed': 'Y11', 'TeamID': 1275}])], ignore_index=True)
seeds_m = pd.concat([seeds_m, pd.DataFrame([{'Season': 2026, 'Seed': 'Y16', 'TeamID': 1224}])], ignore_index=True)
seeds_m = pd.concat([seeds_m, pd.DataFrame([{'Season': 2026, 'Seed': 'Z11', 'TeamID': 1301}])], ignore_index=True)

seeds_w = pd.concat([seeds_w, pd.DataFrame([{'Season': 2026, 'Seed': 'X10', 'TeamID': 3113}])], ignore_index=True)
seeds_w = pd.concat([seeds_w, pd.DataFrame([{'Season': 2026, 'Seed': 'X16', 'TeamID': 3359}])], ignore_index=True)
seeds_w = pd.concat([seeds_w, pd.DataFrame([{'Season': 2026, 'Seed': 'Y16', 'TeamID': 3283}])], ignore_index=True)
seeds_w = pd.concat([seeds_w, pd.DataFrame([{'Season': 2026, 'Seed': 'Z11', 'TeamID': 3304}])], ignore_index=True)

teams = pd.read_csv("data/MTeams.csv")
seeds_m = seeds_m.merge(teams[['TeamID', 'TeamName']], on='TeamID')
seeds_m = seeds_m[seeds_m['Season'] == year]

teams_w = pd.read_csv("data/WTeams.csv")
seeds_w = seeds_w.merge(teams_w[['TeamID', 'TeamName']], on='TeamID')
seeds_w = seeds_w[seeds_w['Season'] == year]

# Determining how many brackets were generated
num_brackets = bracket_sims['Bracket'].max()

# Labeling teams with the percentages of winning a particular matchup, given how many times they won in the simulation
value_counts = m_bracket_sims[m_bracket_sims['Slot'] == matchup_code]['Team'].value_counts()
percentages = (value_counts / num_brackets) * 100
team_names = seeds_m.set_index('Seed')['TeamName']
percentages.index = percentages.index.map(team_names)

# Represents the conditional probability that a team wins a particular matchup
percentages


Team
Duke              35.24
Michigan          20.50
Arizona           17.00
Florida            7.87
Houston            5.58
Iowa St            4.35
Purdue             1.96
Illinois           1.36
Connecticut        1.07
Arkansas           1.02
Michigan St        0.90
Gonzaga            0.87
Virginia           0.85
Kansas             0.30
Alabama            0.25
St John's          0.13
Louisville         0.13
Nebraska           0.12
Texas Tech         0.07
Vanderbilt         0.07
Wisconsin          0.06
Tennessee          0.05
BYU                0.04
UCLA               0.04
Clemson            0.03
Ohio St            0.03
North Carolina     0.03
Miami FL           0.01
VCU                0.01
Texas A&M          0.01
Georgia            0.01
TCU                0.01
Kentucky           0.01
St Mary's CA       0.01
NC State           0.01
Name: count, dtype: float64

In [5]:
# Labeling teams with the percentages of winning a particular matchup, given how many times they won in the simulation
value_counts = w_bracket_sims[w_bracket_sims['Slot'] == matchup_code]['Team'].value_counts()
percentages = (value_counts / num_brackets) * 100
team_names = seeds_w.set_index('Seed')['TeamName']
percentages.index = percentages.index.map(team_names)

# Represents the conditional probability that a team wins a particular matchup
percentages

Team
South Carolina    24.37
UCLA              24.32
Connecticut       22.48
Texas             20.43
LSU                6.95
Michigan           0.67
TCU                0.19
Iowa               0.18
Vanderbilt         0.10
Louisville         0.05
Oklahoma           0.05
Ohio St            0.04
Duke               0.04
North Carolina     0.04
Kentucky           0.03
Notre Dame         0.03
Georgia            0.01
Maryland           0.01
West Virginia      0.01
Name: count, dtype: float64

In [6]:
# Make a copy of bracket_simulations.csv FIRST, name it remaining_perfect_brackets.csv

def count_perfect_brackets(team_id, slot):
    """
    Count the number of perfect brackets remaining for a given team ID and slot.
    
    Parameters:
    team_id (int): The ID of the team.
    slot (str): The slot corresponding to a particular matchup.
    
    Returns:
    int: The number of perfect brackets remaining.
    """
    # Load the remaining perfect brackets
    remaining_brackets = pd.read_csv('remaining_perfect_brackets.csv')
    
    # Filter the bracket simulations for the given slot and team ID
    matching_brackets = m_bracket_sims[(m_bracket_sims['Slot'] == slot) & (m_bracket_sims['Team'] == team_id)]
    
    # Get the list of brackets that have the given team ID and slot
    matching_bracket_ids = matching_brackets['Bracket'].unique()
    
    # Remove brackets that don't have this pick from the remaining perfect brackets
    remaining_brackets = remaining_brackets[remaining_brackets['Bracket'].isin(matching_bracket_ids)]
    
    # Save the updated remaining perfect brackets
    remaining_brackets.to_csv('remaining_perfect_brackets.csv', index=False)
    
    # Return the count of remaining perfect brackets
    return remaining_brackets['Bracket'].nunique()

team_id = ''  # Replace with the desired team seed identifier
slot = 'R1W8'  # Replace with the desired slot
perfect_brackets = 0

try:
    assert(len(team_id) > 0)
    assert(len(slot) > 0)
    perfect_brackets = count_perfect_brackets(team_id, slot)
    print(f"Percent of perfect brackets remaining for team ID {team_id} in slot {slot}: {(perfect_brackets / num_brackets)*100}% ({perfect_brackets} brackets)")
except:
    remaining_brackets = pd.read_csv('remaining_perfect_brackets.csv')
    print(f"Percent of perfect brackets remaining: {(remaining_brackets['Bracket'].nunique() / num_brackets)*100}% ({remaining_brackets['Bracket'].nunique()} brackets)")    
    print()
    perfect_brackets = 0

FileNotFoundError: [Errno 2] No such file or directory: 'remaining_perfect_brackets.csv'